# Prediction Pipeline

This pipeline should run every time I want match predictions. For now the pipeline should do the following:
1. Accept the home squad id and away squad id of the teams playing
2. Get players in squads and their statistics
3. Arrange players into teams
4. Output features
5. Run model on features
6. Display prediction

In [ ]:
# Imports and constants
import numpy as np
import pandas as pd
from helpers.prediction import get_players_statistics, uncondense_parameters, f_x, interpret_probabilities

player_statistics_file = "data/player_statistics_2026-06-13 18:03:37.078636.csv"
squad_members = pd.read_excel('data/World Cup 2026.xlsx', sheet_name="Squad Members")
statistics = pd.read_csv(player_statistics_file)
players = pd.read_excel('data/World Cup 2026.xlsx', sheet_name="Players")

model_weights_concatenated = np.load("parameters/model_V3_parameters_2026-06-12 16:16:55.752520.npy", allow_pickle=True)
mean_std = np.load("parameters/model_v3_mean_std_2026-06-12 16:17:53.057086.npy")

/home/roman/Code/AIEngineering/.venv/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [38]:
def get_team_statistics_from_squad(squad_id: int, match_date: str):
    """ 
    This function should return a (11,9) vector with the statistics of the players in squad provided
    
    Args:
        squad_id (scalar): identifier of squad playing
        match_date (str): Date of the match in YYYY-MM-dd
        
    Returns:
        team_statistics (ndarray): (11,9) vector with statistics of the players from the team
    """
    team_members = squad_members[squad_members["squad_id"] == squad_id].head(n=11)
    player_ids = team_members['player_id']
    if len(player_ids) != 11:
        raise Exception("Could not get players for team")
    
    player_ids = player_ids.tolist()
    team = np.empty((0,9))
    for p_id in player_ids:
        stats = get_players_statistics(players=players, statistics=statistics, player_id=p_id, match_date=match_date)
        team = np.append(team, stats, axis=0)
        
    return team

squad_players = get_team_statistics_from_squad(1, "2026-06-11")
print(squad_players)

[[ 26.           0.           0.           0.           1.
    0.         144.           0.           3.        ]
 [ 26.           0.           1.           0.           0.
    1.         197.           9.           1.75      ]
 [ 29.           0.           0.           1.           0.
    2.         300.          16.           3.25      ]
 [ 27.           0.           0.           1.           0.
    1.         249.          10.           2.125     ]
 [ 31.           0.           0.           1.           0.
    2.         336.          41.           4.        ]
 [ 26.           0.           1.           0.           0.
    1.         224.           1.           4.        ]
 [ 29.           0.           1.           0.           0.
    0.         337.          35.           4.        ]
 [ 25.           1.           0.           0.           0.
    1.         191.          70.           3.        ]
 [ 27.           0.           1.           0.           0.
    2.         347.          

In [39]:
def get_feature_to_predict(
    home_squad_id: int,
    away_squad_id: int,
    match_date: str
):
    """ 
    Get feature that I can use to make a prediction
    
    Args:
        home_squad_id: Identifer of the squad for the home team
        away_squad_id: Identifer of the squad for the away team
        match_date: Date match is played in YYYY-MM-dd
        
    Returns:
        x (ndarray): a (18,1) array that has also been normalized
    """
    home_team = get_team_statistics_from_squad(squad_id=home_squad_id, match_date=match_date)
    away_team = get_team_statistics_from_squad(squad_id=away_squad_id, match_date=match_date)
    
    # Get sum of all members in each team
    sum_home_team = np.sum(home_team, axis=0)
    sum_away_team = np.sum(away_team, axis=0)
    
    # Concatenate
    sum_concated = (np.concatenate((sum_home_team, sum_away_team), axis=-1)).reshape((18, 1))
    
    # Normalize
    mean = mean_std[0]
    std = mean_std[0]
    
    sum_normalized = (sum_concated - mean) / std
    
    return sum_normalized
    
get_feature_to_predict(home_squad_id=1, away_squad_id=2, match_date="2026-06-11")

array([[-0.1162319 ],
       [-0.99144741],
       [-0.98859654],
       [-0.99144741],
       [-0.99714914],
       [-0.96293876],
       [ 7.64952388],
       [ 0.40832722],
       [-0.8991846 ],
       [-0.13618796],
       [-0.99429827],
       [-0.99144741],
       [-0.98574568],
       [-0.99714914],
       [-0.96293876],
       [ 4.06028506],
       [-0.44693222],
       [-0.89416956]])

In [43]:
def make_game_prediction(home_squad: int, away_squad: int, match_date: int):
    """  
    Make prediction of outcome between home squad and away squad using weights of learnt model
    
    Args:
        home_squad (scalar): ID of home squad
        away_squad (scalar): ID of away squad
        match_date (scalar): Date match is being played in YYYY-MM-dd
    """
    feature = get_feature_to_predict(home_squad_id=home_squad, away_squad_id=away_squad, match_date=match_date)
    W1, b1, W2, b2, W3, b3 = uncondense_parameters(model_weights_concatenated)
    pred, _, _, _, _ = f_x(X=feature, W1=W1, W2=W2, W3=W3, b1=b1, b2=b2, b3=b3)
    prediction = interpret_probabilities(pred)
    print(prediction)
    
make_game_prediction(7,8, "2026-06-11")
    

3-0
